# 09 - Cheap wins: CCI, ECE_atk, attack-class Brier, decoupling
All from cached scores, no new model runs. Closes three reviewer objections:
1. **CCI (Calibration Collapse Index)** - relative change vs in-distribution. Makes 'collapse' a number.
2. **ECE_atk + attack-class Brier** vs **pooled ECE** - answers 'isn't S just ECE on attacks?' and shows pooled calibration looks fine while attack-conditional is broken.
3. **Decoupling** - S carries information not in FNR/AUROC/ECE_atk, so it is not redundant.

## Bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first) ===
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')
!git pull
print('Session ready.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready.


## Reconstruct per-(detector, shift) (y, p) from cached scores

In [2]:
import importlib, data_loaders, detectors, metrics
for m in (data_loaders, detectors, metrics): importlib.reload(m)
from data_loaders import load_deepset, load_bipia_categorized, load_notinject, load_jailbreak, BIPIA_HARMFUL, BIPIA_HIJACK, BIPIA_MALICIOUS
from detectors import threshold_at_fpr, auroc
from metrics import fnr, severity_S, ece_pooled, ece_attack_conditional
import numpy as np, pandas as pd, os

deepset=load_deepset(); bip=load_bipia_categorized(per_category=30,categories=BIPIA_MALICIOUS); noti=load_notinject()
try: jb=load_jailbreak()
except Exception as e: jb=None; print('[no jailbreak]',e)
yd=deepset.label.values
DET={'protectai_v2':'data/score_protectai_v2','prompt_guard_2':'data/score_prompt_guard_2','prompt_guard_2_22m':'data/score_prompt_guard_2_22m'}

def brier_atk(y,p):
    y=np.asarray(y); p=np.asarray(p); a=y==1
    return float(np.mean((1-p[a])**2)) if a.sum() else np.nan

def cells_for(base):
    d=np.load(base+'_deepset.npy'); b=np.load(base+'_bipia_cat.npy')
    out={'direct':(yd,d)}
    bl=bip.label.values==0; H=bip.meta.isin(BIPIA_HARMFUL).values; J=bip.meta.isin(BIPIA_HIJACK).values
    out['indirect_harmful']=(np.r_[np.ones(H.sum()),np.zeros(bl.sum())], np.r_[b[H],b[bl]])
    out['indirect_hijack'] =(np.r_[np.ones(J.sum()),np.zeros(bl.sum())], np.r_[b[J],b[bl]])
    if jb is not None and os.path.exists(base+'_jailbreak.npy'):
        out['jailbreak']=(jb.label.values, np.load(base+'_jailbreak.npy'))
    if os.path.exists(base+'_notinject.npy'):
        out['over_defense']=(noti.label.values, np.load(base+'_notinject.npy'))
    return out
print('ready')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

...

jailbreak_dataset_train_balanced.csv:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

jailbreak_dataset_test_balanced.csv:   0%|          | 0.00/370k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1044 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/262 [00:00<?, ? examples/s]

ready


## Panel with ECE_atk, pooled ECE, Brier

In [3]:
rows=[]
for det,base in DET.items():
    if not os.path.exists(base+'_deepset.npy'): print('missing',det); continue
    t=threshold_at_fpr(np.load(base+'_deepset.npy')[yd==0],0.01)
    for sh,(y,p) in cells_for(base).items():
        rows.append({'detector':det,'shift':sh,
            'FNR':round(fnr(y,p,t),3),'S':round(severity_S(y,p,t),3),
            'ECE_atk':round(ece_attack_conditional(y,p,t),3),'ECE_pooled':round(ece_pooled(y,p,t),3),
            'Brier_atk':round(brier_atk(y,p),3),'AUROC':(round(auroc(y,p),3) if len(np.unique(y))>1 else np.nan)})
panel=pd.DataFrame(rows)
print(panel.to_string(index=False))

          detector            shift   FNR     S  ECE_atk  ECE_pooled  Brier_atk  AUROC
      protectai_v2           direct 0.548 0.999    0.041       0.238      0.579  0.882
      protectai_v2 indirect_harmful 0.650 0.995    0.109       0.188      0.730  0.444
      protectai_v2  indirect_hijack 0.693 0.993    0.157       0.239      0.821  0.424
      protectai_v2        jailbreak 0.136 0.995    0.032       0.087      0.158  0.986
      protectai_v2     over_defense   NaN 0.000      NaN       0.424        NaN    NaN
    prompt_guard_2           direct 0.532 0.999    0.234       0.302      0.750  0.942
    prompt_guard_2 indirect_harmful 0.217 0.998    0.690       0.062      0.893  0.894
    prompt_guard_2  indirect_hijack 0.693 0.998    0.303       0.159      0.993  0.625
    prompt_guard_2        jailbreak 0.010   NaN    0.049       0.028      0.049  0.993
    prompt_guard_2     over_defense   NaN 0.000      NaN       0.050        NaN    NaN
prompt_guard_2_22m           direct 0.837 0

## ECE_pooled vs ECE_atk: pooled hides what attack-conditional shows

In [4]:
comp=panel[panel['shift'].isin(['indirect_harmful','indirect_hijack'])][['detector','shift','ECE_pooled','ECE_atk','S']]
print(comp.to_string(index=False))
print('\nmean pooled ECE on indirect:', round(comp.ECE_pooled.mean(),3),
      '| mean ECE_atk:', round(comp.ECE_atk.mean(),3))
print('-> pooled calibration looks far healthier than attack-conditional; that gap is the point.')

          detector            shift  ECE_pooled  ECE_atk     S
      protectai_v2 indirect_harmful       0.188    0.109 0.995
      protectai_v2  indirect_hijack       0.239    0.157 0.993
    prompt_guard_2 indirect_harmful       0.062    0.690 0.998
    prompt_guard_2  indirect_hijack       0.159    0.303 0.998
prompt_guard_2_22m indirect_harmful       0.066    0.082 0.996
prompt_guard_2_22m  indirect_hijack       0.159    0.029 0.996

mean pooled ECE on indirect: 0.146 | mean ECE_atk: 0.228
-> pooled calibration looks far healthier than attack-conditional; that gap is the point.


## CCI (Calibration Collapse Index): relative change vs in-distribution

In [5]:
cci=[]
for det in panel.detector.unique():
    d=panel[panel.detector==det]; base=d[d['shift']=='direct'].iloc[0]
    for _,r in d.iterrows():
        if r['shift'] in ('direct','over_defense'): continue
        def rel(c):
            b=base[c]; v=r[c]
            return round((v-b)/b,2) if (pd.notna(b) and b not in (0,) and pd.notna(v)) else np.nan
        cci.append({'detector':det,'shift':r['shift'],'CCI_FNR':rel('FNR'),'CCI_AUROC':rel('AUROC'),
                    'CCI_ECEatk':rel('ECE_atk'),'CCI_S':rel('S')})
cci=pd.DataFrame(cci)
print(cci.to_string(index=False))
print('\nCCI_S ~ 0 (severity does not collapse, it is already saturated) while CCI_AUROC/FNR move sharply: the collapse is in discrimination, not in miss-confidence.')

          detector            shift  CCI_FNR  CCI_AUROC  CCI_ECEatk  CCI_S
      protectai_v2 indirect_harmful     0.19      -0.50        1.66  -0.00
      protectai_v2  indirect_hijack     0.26      -0.52        2.83  -0.01
      protectai_v2        jailbreak    -0.75       0.12       -0.22  -0.00
    prompt_guard_2 indirect_harmful    -0.59      -0.05        1.95  -0.00
    prompt_guard_2  indirect_hijack     0.30      -0.34        0.29  -0.00
    prompt_guard_2        jailbreak    -0.98       0.05       -0.79    NaN
prompt_guard_2_22m indirect_harmful     0.08      -0.11       -0.10   0.00
prompt_guard_2_22m  indirect_hijack     0.16      -0.25       -0.68   0.00
prompt_guard_2_22m        jailbreak    -0.90       0.23        2.09  -0.01

CCI_S ~ 0 (severity does not collapse, it is already saturated) while CCI_AUROC/FNR move sharply: the collapse is in discrimination, not in miss-confidence.


## Decoupling: is S redundant with FNR / AUROC / ECE_atk?

In [6]:
est=panel[panel['shift']!='over_defense'].dropna(subset=['S','FNR','AUROC','ECE_atk'])
print('across', len(est), 'estimable cells:')
for col in ['FNR','AUROC','ECE_atk','Brier_atk']:
    r=np.corrcoef(est['S'],est[col])[0,1]
    print(f'  corr(S, {col}) = {r:+.3f}')
print('\nvariance: S =', round(est.S.var(),5), '| FNR =', round(est.FNR.var(),4), '| AUROC =', round(est.AUROC.var(),4))
print('-> S is near-constant and uncorrelated with the accuracy metrics: not recoverable from them, not redundant.')

across 11 estimable cells:
  corr(S, FNR) = +0.276
  corr(S, AUROC) = +0.149
  corr(S, ECE_atk) = +0.134
  corr(S, Brier_atk) = +0.456

variance: S = 1e-05 | FNR = 0.0927 | AUROC = 0.0419
-> S is near-constant and uncorrelated with the accuracy metrics: not recoverable from them, not redundant.


## Persist + commit

In [7]:
from reslog import log_result
os.makedirs('reports',exist_ok=True)
panel.to_csv('reports/panel_extended_metrics.csv',index=False); cci.to_csv('reports/cci.csv',index=False)
body='EXTENDED PANEL (ECE_atk/pooled/Brier):\n'+panel.to_string(index=False)+'\n\nCCI:\n'+cci.to_string(index=False)
log_result('Cheap-wins: CCI + ECE_atk + Brier + decoupling', body, csv_df=panel, csv_name='panel_extended_metrics.csv')
!git add -A && git commit -m "Cheap-wins: CCI, ECE_atk vs pooled ECE, attack-class Brier, S decoupling" && git push
print('done')

[reslog] appended 'Cheap-wins: CCI + ECE_atk + Brier + decoupling' to reports/RESULTS_LOG.md + panel_extended_metrics.csv
[main f7175b3] Cheap-wins: CCI, ECE_atk vs pooled ECE, attack-class Brier, S decoupling
 5 files changed, 65 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/09_cheap_wins.ipynb
 create mode 100644 reports/cci.csv
 create mode 100644 reports/panel_extended_metrics.csv
Enumerating objects: 14, done.
Counting objects: 100% (14/14), done.
Delta compression using up to 8 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 15.45 KiB | 687.00 KiB/s, done.
Total 9 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 4 local objects.
To https://github.com/anasbiswas1/picalib-research.git
   acaff3e..f7175b3  main -> main
done
